# ALS baseline - MovieLens 100k

In [14]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "als").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "als" / "src"))

from datasets import ensure_movielens
from spark_utils import get_spark_session, stop_spark
from split_utils import load_ml100k_predefined, clean_cold_start
from als_utils import run_als
from results_utils import append_result, new_run_id

In [15]:
# experiment config
DATASET      = "ml-100k"
RESULTS_CSV  = str(PROJECT_ROOT / "results" / "als_results.csv")

K            = 5
LAMBDA       = 0.1
MAX_ITER     = 10
SEED         = 42

# cores to test
CORE_VALUES  = [1, 2, 4]
NUM_RUNS     = 3

DATA_DIR = str(ensure_movielens(DATASET, PROJECT_ROOT / "data"))


In [16]:
spark = get_spark_session("als_100k_prep", num_cores=CORE_VALUES[0])

train, test = load_ml100k_predefined(spark, DATA_DIR)
test_clean = clean_cold_start(train, test)

train_size   = train.count()
test_size    = test_clean.count()
dataset_size = train_size + test_size
print("train:", train_size, "| test (clean):", test_size, "| total:", dataset_size)

stop_spark(spark)

train: 90570 | test (clean): 9428 | total: 99998


core-scaling runs


In [17]:
for num_cores in CORE_VALUES:
    print(f"{num_cores} core(s)")
    spark = get_spark_session(f"als_100k_{num_cores}c", num_cores=num_cores)

    train, test = load_ml100k_predefined(spark, DATA_DIR)
    test_clean = clean_cold_start(train, test).cache()
    train = train.cache()
    train.count(); test_clean.count()

    for run in range(1, NUM_RUNS + 1):
        row = run_als(
            train, test_clean,
            dataset=DATASET, dataset_size=dataset_size, num_cores=num_cores,
            k=K, lam=LAMBDA, max_iter=MAX_ITER, seed=SEED,
            run_id=new_run_id(),
        )
        append_result(row, RESULTS_CSV)
        print(f"  run {run}/{NUM_RUNS}: RMSE={row['test_rmse']:.4f}  train_time={row['train_time']:.2f}s")

    stop_spark(spark)

print("Results appended to", RESULTS_CSV)

1 core(s)
  run 1/3: RMSE=0.9483  train_time=1.27s
  run 2/3: RMSE=0.9483  train_time=1.23s
  run 3/3: RMSE=0.9483  train_time=1.23s
2 core(s)
  run 1/3: RMSE=0.9483  train_time=0.83s
  run 2/3: RMSE=0.9483  train_time=0.84s
  run 3/3: RMSE=0.9483  train_time=0.84s
4 core(s)
  run 1/3: RMSE=0.9483  train_time=0.71s
  run 2/3: RMSE=0.9483  train_time=0.69s
  run 3/3: RMSE=0.9483  train_time=0.67s
Results appended to c:\Users\89526\Documents\GitHub\ccdpp-pyspark-als-movielens\results\als_results.csv


## Inspect results

In [ ]:
import pandas as pd
df = pd.read_csv(RESULTS_CSV)
summary = (df[df.dataset == DATASET]
           .groupby("num_cores")
           .agg(mean_rmse=("test_rmse","mean"),
                mean_train_time=("train_time","mean"),
                runs=("run_id","count"))
           .reset_index())
base = summary.loc[summary.num_cores == summary.num_cores.min(), "mean_train_time"].iloc[0]
summary["speedup"]    = base / summary["mean_train_time"]
summary["efficiency"] = summary["speedup"] / summary["num_cores"]
summary

,num_cores,mean_rmse,mean_train_time,runs,speedup,efficiency
0,1,0.948341,1.545845,3,1.000000,1.000000
1,2,0.948341,1.028234,4,1.503398,0.751699
2,4,0.948341,0.728513,3,2.121920,0.530480
